### 주제

온라인 채널 제품 판매량 예측

### 데이터

- **train.csv [파일]**
    - ID : 실제 판매되고 있는 고유 ID
    - 제품 : 제품 코드
    - 대분류 : 제품의 대분류 코드
    - 중분류 : 제품의 중분류 코드
    - 소분류 : 제품의 소분류 코드
    - 브랜드 : 제품의 브랜드 코드
    - 쇼핑몰 : 쇼핑몰 코드
    - 2022-01-01 ~ 2023-04-24 : 실제 일별 판매량
    - 단, 제품이 동일하여도 판매되고 있는 고유 ID 별로 기재한 분류 정보가 상이할 수 있음
    - 즉 고유 ID가 다르다면, 제품이 같더라도 다른 판매 채널
- **sample_submission.csv [파일] - 제출 양식**
    - ID : 실제 판매되고 있는 고유 ID
    - 2023-04-25 ~ 2023-05-15 : 예측한 일별 판매량
- **sales.csv [파일] - 메타(Meta) 정보**
    - ID : 실제 판매되고 있는 고유 ID
    - 제품 : 제품 코드
    - 대분류 : 제품의 대분류 코드
    - 중분류 : 제품의 중분류 코드
    - 소분류 : 제품의 소분류 코드
    - 브랜드 : 제품의 브랜드 코드
    - 쇼핑몰 : 쇼핑몰 코드
    - 2022-01-01 ~ 2023-04-24 : 실제 일별 총 판매금액
    - 단, 제품이 동일하여도 판매되고 있는 고유 ID 별로 기재한 분류 정보가 상이할 수 있음
    - 즉 고유 ID가 다르다면, 제품이 같더라도 다른 판매 채널
- **brand_keyword_cnt.csv [파일] - 메타(Meta) 정보**
    - 브랜드 : 브랜드 코드
    - 2022-01-01 ~ 2023-04-24 : 브랜드의 연관키워드 언급량을 정규화한 일별 데이터
- **product_info.csv [파일] - 메타(Meta) 정보**
    - 제품 : 제품 코드
    - 제품특성 : 제품 특성 데이터(Text)
    - train.csv에 존재하는 모든 제품 코드가 포함되어 있지 않음. 또는 product_info.csv에 존재하는 제품 코드가 train.csv에 존재하지 않을 수 있음

### 코드흐름

1. 데이터 로드
    - train.csv
    - brand_keyword_cnt.csv
    - sales.csv
2. 전처리
    - 브랜드 키워드 데이터를 train에 merge
    - 날짜별 feature 확장 (wide 형태)
    - LabelEncodder로 범주형 처리
3. 시계열 데이터 생성
    - siding window 방식
    - 입력: 90일
    - 출력: 21일
    - 구조
        - X: (batch, 90, feature)
        - y: (batch, 21)

In [ ]:
def make_sequence_data(data, window_size=90, pred_size=21):
    X, y = [], []

    for i in range(len(data) - window_size - pred_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size:i+window_size+pred_size])

    return np.array(X), np.array(y)

4. Dataset & DataLoader
    - Pytorch Dataset 커스텀 정의
    - DataLoader로 batch 처리
    - 구조
        - Dataset → (X, y) 변환
        - DataLoader → batch 학습
5. LSTM 모델 정의
    - 구조
        - Input → LSTM → FC → Output
    - sequence input 처리
    - 마지막 hidden state 기반 예측

In [ ]:
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # 마지막 시점
        out = self.fc(out)
        return out

6. Custom Loss
    - 특정 구간 더 중요하게
    - 피크 값 강조
    - 시계열 shape 유지
7. 학습 루프

In [ ]:
for epich:
	for batch:
			forward
			loss 계산
			backward
			optimizer.step()

  - batch size 큼
  - GPU 활용
8. 예측
    - test 데이터를 동일하게 window로 변환
    - 모델로 21일 예측 생성
9. Max Ensemble
    - 여러 결과 앙상블

In [ ]:
final_pred = np.maximum(pred1, pred2)

### 새롭게 알게 된 내용 / 어려운 내용 / 배울 점

- 시계열을 지도학습으로 바꾸는 방식
    - sliding window를 통해 과거 → 미래 예측 구조로 변환하는 방식을 이해함
    - 단순 시계열이 아니라 (X:90일, y:21일) 형태로 모델에 넣는 게 핵심
    - 시계열을 RNN이 아니라 지도학습으로 변환 가능
- Multi-step 예측 구조
    - 하루씩 예측하는 게 아니라 21일을 한 번에 예측하는 방식
    - LSTM의 마지막 hidden state를 활용해서 여러 시점을 동시에 출력
- Max Ensemble 전략
    - 평균이 아니라 max를 사용하는 앙상블
    - 특히 수요 예측에서 과소 예측 방지에 효과적